# Phase 5 - Scenario Simulation and Capacity-Pressure Analysis

This notebook reviews the scenario simulation outputs created by `src/scenario_simulation.py`. The purpose is to estimate demand ranges and capacity-threshold exceedance risk under simplified stress assumptions. This is a decision-support stress-testing layer, not a physical electricity grid dispatch model.

## Forecast-Design Context

The default base forecast is `gradient_boosting_forecast` from the Phase 4 feature-model output. It should be interpreted as an operational one-day-ahead forecast baseline because it can use recent actual demand history. Strict recursive validation showed that seasonal naive remained slightly stronger for long-horizon forecasting, so the scenario layer should not be presented as a full-year-ahead prediction of actual grid stress.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.scenario_simulation import run_scenario_simulation, TABLES_DIR, SCENARIO_FIGURES_DIR

TARGET = 'nd_mean'
N_SIMULATIONS = 1000
RANDOM_SEED = 42

## Run Scenario Simulation

Run the simulation after the feature modelling and forecast validation outputs have been generated locally.

In [ ]:
daily_summary, scenario_summary, threshold_documentation = run_scenario_simulation(
    target=TARGET,
    n_simulations=N_SIMULATIONS,
    random_seed=RANDOM_SEED,
)

threshold_documentation

## Scenario-Level Capacity Pressure

The scenario-level summary ranks scenarios by expected capacity-threshold exceedance days and highlights how concentrated the risk is.

In [ ]:
scenario_summary.sort_values('expected_exceedance_days', ascending=False)

## Daily Simulation Output

The daily output includes the base forecast, simulated range and exceedance probability for each date and scenario.

In [ ]:
daily_summary.head()

## Highest-Risk Scenario

The highest-risk scenario is the one with the largest expected number of exceedance days. This is useful for comparing stress assumptions, but it should not be interpreted as a precise probability of real-world grid failure.

In [ ]:
highest_risk = scenario_summary.sort_values('expected_exceedance_days', ascending=False).head(1)
highest_risk

## Figures

The script saves figures to `outputs/figures/scenario_simulation/`.

In [ ]:
for figure_name in [
    'scenario_exceedance_probability.png',
    'scenario_simulated_demand_ranges.png',
    'scenario_capacity_pressure_summary.png',
]:
    figure_path = SCENARIO_FIGURES_DIR / figure_name
    if figure_path.exists():
        display(Image(filename=str(figure_path)))
    else:
        print(f'Missing figure: {figure_path}')

## Limitations

- Scenarios are simplified stress tests.
- Exogenous effects are proxy assumptions, especially the low-renewable stress case.
- Results depend on the forecast design and base forecast used.
- The default base forecast is operational one-day-ahead, not a full-year-ahead forecast.
- This is not a physical grid dispatch model.
- Forecast evaluation remains separate from scenario simulation.